# Imports

These magic commands allow us to reload libraries without having to restart the kernel.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tree_propagation.consts import DEFAULT_RANDOM_SEED, DEFAULT_HEIGHT_CLASS
from tree_propagation.environment import DEFAULT_ENVIRONMENT_STATE
from tree_propagation.metrics import evaluate_seed_shadow
from tree_propagation.simulation import SeedDispersalSimulation, SeedDispersalSimulationParams, DEFAULT_SEED_DISPERSAL_SIMULATION_INSTANCES
from tree_propagation.species_config import load_species_configs_from_csv
from tree_propagation.visualization.matplot import plot_simulated_seed_shadow_and_distance_histogram

# General configuration

In [3]:
logging.basicConfig(level=logging.INFO)

# Simulation configuration

RNG is set with a static default seed so that stochastic processes become reproducible.

In [4]:
rng = np.random.default_rng(DEFAULT_RANDOM_SEED)

The species configuration is loaded from the file.

In [5]:
species_config = load_species_configs_from_csv("./data/tree_species_config.csv")

For each species configuration, we execute a Monte Carlo simulation of seed dispersal using the default number of instances. The results are then measured and converted into rows, and we generate a side by side plot of the seed shadow and the histogram of distances, and how these compare to the expected mean. The plots are then saved to the overleaf directory for inclusion in the report.

In [6]:
MAX_EXTENT = 500.0  # Cap 2D seed shadow at ±500 m (>98% of Pappel seeds)
MAX_DISTANCE = None #700.0 # Cap histogram

rows = []
for cfg in species_config.values():
    params = SeedDispersalSimulationParams(
        config=cfg, environment_state=DEFAULT_ENVIRONMENT_STATE, height_class=DEFAULT_HEIGHT_CLASS, num_instances=DEFAULT_SEED_DISPERSAL_SIMULATION_INSTANCES
    )
    result = SeedDispersalSimulation(params, rng=rng).run()
    m = evaluate_seed_shadow(result)
    rows.append({
        "species": cfg.species_name,
        "dispersal": cfg.dispersal_type.value,
        "total_seeds": m.total_seeds,
        "r_bar_theory_m": m.theoretical_mean_radius,
        "r_bar_sim_m": m.empirical_mean_radius,
        "rel_error_pct": m.rel_error_pct,
        "ks_pvalue": m.ks_pvalue,
    })
    fig = plot_simulated_seed_shadow_and_distance_histogram(
        result=result,
        size=(10, 4.8),
        max_extent=MAX_EXTENT,
        max_dist=MAX_DISTANCE,
    )
    fig.savefig(
        f"docs/overleaf/images/seed_shadow_{cfg.species_name.replace('ü', 'ue')}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.close(fig)

The results are converted into a Pandas dataframe and writen to the overleaf directory for inclusion into the report in tabular format.

In [7]:
df_results = pd.DataFrame(rows).round(3)
display(df_results)

seed_dispersal_mc_results = df_results.to_latex(
    index=False,
    float_format="%.2f",
    na_rep="—",
)

with open("./docs/overleaf/tables/seed_dispersal_montecarlo_results.tex", "w", encoding="utf-8") as f:
    f.write(seed_dispersal_mc_results)

df_results.to_csv("./docs/overleaf/tables/seed_dispersal_montecarlo_results.csv", index=False)

,species,dispersal,total_seeds,r_bar_theory_m,r_bar_sim_m,rel_error_pct,ks_pvalue
0,Ahorn,wind,8066,36.0,35.689,0.863,0.343
1,Akazie,gravity,12005,10.0,9.910,0.904,0.301
2,Buche,animal,4851,30.0,31.146,3.821,0.116
3,Esche,wind,9538,28.0,28.181,0.648,0.897
4,Kastanie,animal,3095,10.0,9.439,5.614,0.108
5,Kiefer,wind,7989,44.0,43.841,0.361,0.605
6,Kirsche,animal,9656,60.0,59.870,0.217,0.390
7,Linde,wind,6461,28.0,28.012,0.044,0.453
8,Pappel,wind,8154,120.0,121.757,1.464,0.455
9,Platane,wind,6369,72.0,73.269,1.763,0.065
